In [0]:

SHOW TABLES IN pondolens.dev_bronze;

### 1. Check schema

In [0]:
DESCRIBE pondolens.dev_bronze.grdp;

### 2. Compare overall rows versus 2021–2025

In [0]:
SELECT
    COUNT(*) AS overall_rows,
    COUNT_IF(TRY_CAST(year AS INT) BETWEEN 2021 AND 2025)
        AS rows_2021_2025,
    COUNT_IF(
        TRY_CAST(year AS INT) < 2021
        OR TRY_CAST(year AS INT) > 2025
    ) AS rows_outside_scope,
    COUNT_IF(TRY_CAST(year AS INT) IS NULL)
        AS missing_or_unparseable_years
FROM pondolens.dev_bronze.grdp;

### 3. Check rows and geography count per year

In [0]:
SELECT
    year,
    geographic_level,
    COUNT(*) AS row_count,
    COUNT(DISTINCT region_name) AS geography_count
FROM pondolens.dev_bronze.grdp
GROUP BY year, geographic_level
ORDER BY year, geographic_level;

### 4. Check actual geography labels, including NIR and national total

In [0]:
SELECT
    region_name,
    geographic_level,
    COUNT(*) AS row_count,
    COUNT(DISTINCT year) AS years_present
FROM pondolens.dev_bronze.grdp
GROUP BY region_name, geographic_level
ORDER BY geographic_level, region_name;

### 5. Check source versions

In [0]:
SELECT
    source_file,
    source_release,
    _dataset_hash,
    _run_id,
    COUNT(*) AS row_count
FROM pondolens.dev_bronze.grdp
GROUP BY
    source_file,
    source_release,
    _dataset_hash,
    _run_id
ORDER BY source_release, _run_id;

## Final findings

The completed profiling checks cover 95 records for 2021–2025: 18 regions and one national total per year, including Negros Island Region.

- No missing required values, invalid years or amounts, zero amounts, duplicate keys, missing geography-year combinations, label issues, rescued rows, or missing checked lineage fields were reported.
- Regional sums match national totals for both GRDP measures in all five years.
- All 152 computed growth values match the original workbook at two decimal places: 19 geographies × 4 annual comparisons × 2 measures.
- Western Visayas 2022 exceeded the proposed 20% review threshold: 25.22% current-price growth and 10.23% constant-price growth. Both match the source; retain the record. No quarantine is required for this finding.
- Growth for 2021 is not applicable because 2020 is outside the ingested scope.

Source units are **thousand Philippine pesos**. The source assigns Sulu to Zamboanga Peninsula throughout 2018–2025. Confirm regional boundaries before joining with population or budget data.

These findings summarize the completed checks and source comparison; the detailed temporary-view queries have been removed.


## Proposed Silver columns

**Grain:** one row per geography, year, and source release.

**Primary key:** `grdp_id`, a unique, stable ID such as `0001` or `0002`. Preserve existing IDs on reruns.

**Business key:** `(region_code, year, source_release)` must also be unique and non-null.

| Column | Proposed type | Purpose |
|---|---|---|
| `grdp_id` | STRING | Unique, stable record ID |
| `region_code` | STRING | Verified geography code, including the national total |
| `region_name` | STRING | Standardized geography name |
| `geographic_level` | STRING | `region` or `national` |
| `year` | INT | Reporting year |
| `grdp_current_thousand_php` | DECIMAL(p, s) | GRDP at current prices, in thousand PHP |
| `grdp_constant_2018_thousand_php` | DECIMAL(p, s) | GRDP at constant 2018 prices, in thousand PHP |
| `boundary_version` | STRING | Verified source geographic definition |
| `source_release` | STRING | Publication version, e.g. `2026-04` |
| `source_file` | STRING | Original source filename |
| `drive_file_id` | STRING | Ingested Google Drive file ID |
| `run_id` | STRING | Bronze ingestion run ID |
| `ingested_at` | TIMESTAMP | Original Bronze ingestion timestamp |

Confirm decimal precision and scale before casting. Use verified mappings for geography codes and boundaries.

Keep `dataset_hash` in Bronze and quality-review results in a separate table. Calculate growth in Gold.

Retain national rows, but exclude them when summing regional amounts. Record Western Visayas 2022 as `SOURCE_VERIFIED_LARGE_CHANGE` in the review table.